In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import fiona
import geopandas as gpd

import sys
from pathlib import Path
# -------------------------------------------------------------------
# Define the base data directory (relative to this script's location)
# -------------------------------------------------------------------
DATA_DIR = Path.cwd().parent / "data"
print("Data directory:", DATA_DIR.resolve())

Data directory: C:\Users\cort3\Documents\Classes\OptimalChargerPlacement\data


# Import Data

In [19]:
# Import load data


def import_allFeeder_loadDataDf():
    file_path = DATA_DIR / "cleanedGridData" / "feeder_load_profile_clean.csv"
    df_load = pd.read_csv(file_path)

    # sort by feeder_id and timestamp
    df_load = df_load.sort_values(by=['feeder_id', 'month', 'hour']) # .reset_index()
    return df_load

def get_loadDf_forFeeder(feeder_id):
    df_load = import_allFeeder_loadDataDf()
    df_feeder = df_load[df_load['feeder_id'] == int(feeder_id)].copy()
    return df_feeder

loadDf = import_allFeeder_loadDataDf()
display(loadDf.head())

df_feeder = get_loadDf_forFeeder("012011109")
display(df_feeder.head())

,feeder_id,low_load_kw,high_load_kw,month,hour
36288,12011109,4294.0,4832.0,1,0
36289,12011109,4230.0,4708.0,1,1
36290,12011109,4152.0,4633.0,1,2
36291,12011109,4157.0,4602.0,1,3
36292,12011109,4148.0,4604.0,1,4


,feeder_id,low_load_kw,high_load_kw,month,hour
36288,12011109,4294.0,4832.0,1,0
36289,12011109,4230.0,4708.0,1,1
36290,12011109,4152.0,4633.0,1,2
36291,12011109,4157.0,4602.0,1,3
36292,12011109,4148.0,4604.0,1,4


In [ ]:
# import meta data for all feed
# Import metadata geodataframe for all feeders
# 1 row for each feeder
# 'feeder_id'	'division'	'substation'	'nom_volt_kV'	'Existing_DG'	'Queued_DG'	'shape_length'	'geometry'


def import_metaData_geodataframe_all_feeders():
    file_path = DATA_DIR / "cleanedGridData" / "feeder_meta_clean_gdf.gpkg"
    gdf_all_feeders = gpd.read_file(file_path)
    # sort by feeder_id
    gdf_all_feeders = gdf_all_feeders.sort_values(by=['feeder_id']).reset_index()
    # remove old index column
    gdf_all_feeders = gdf_all_feeders.drop(columns=['index'])
    display(gdf_all_feeders)
    return gdf_all_feeders


import_metaData_geodataframe_all_feeders()

,feeder_id,division,substation,nom_volt_kV,Existing_DG,Queued_DG,shape_length,geometry
0,012011101,East Bay,OAKLAND C,12kV,0,0,3987.047096,"MULTILINESTRING ((564059.182 4184096.28, 56405..."
1,012011102,East Bay,OAKLAND C,12kV,0,0,4937.485621,"MULTILINESTRING ((564557.517 4184407.182, 5645..."
2,012011103,East Bay,OAKLAND C,12kV,0,40,3925.468328,"MULTILINESTRING ((564081.619 4184244.407, 5640..."
3,012011104,East Bay,OAKLAND C,12kV,0,350,4467.524968,"MULTILINESTRING ((564235.877 4184316.718, 5642..."
4,012011105,East Bay,OAKLAND C,12kV,0,0,3778.285873,"MULTILINESTRING ((564381.798 4184558.433, 5643..."
...,...,...,...,...,...,...,...,...
3018,255391102,Yosemite,BONITA,12kV,6590,590,64498.898947,"MULTILINESTRING ((758390.816 4094829.522, 7583..."
3019,255391103,Yosemite,BONITA,12kV,3742,3842,63686.393390,"MULTILINESTRING ((749035.452 4094939.762, 7490..."
3020,255451102,Kern,CAL WATER,12kV,10520,370,97943.786622,"MULTILINESTRING ((876103.782 3926133.744, 8762..."
3021,255451103,Kern,CAL WATER,12kV,10380,300,72464.525600,"MULTILINESTRING ((874170.252 3926169.744, 8741..."


,feeder_id,division,substation,nom_volt_kV,Existing_DG,Queued_DG,shape_length,geometry
0,012011101,East Bay,OAKLAND C,12kV,0,0,3987.047096,"MULTILINESTRING ((564059.182 4184096.28, 56405..."
1,012011102,East Bay,OAKLAND C,12kV,0,0,4937.485621,"MULTILINESTRING ((564557.517 4184407.182, 5645..."
2,012011103,East Bay,OAKLAND C,12kV,0,40,3925.468328,"MULTILINESTRING ((564081.619 4184244.407, 5640..."
3,012011104,East Bay,OAKLAND C,12kV,0,350,4467.524968,"MULTILINESTRING ((564235.877 4184316.718, 5642..."
4,012011105,East Bay,OAKLAND C,12kV,0,0,3778.285873,"MULTILINESTRING ((564381.798 4184558.433, 5643..."
...,...,...,...,...,...,...,...,...
3018,255391102,Yosemite,BONITA,12kV,6590,590,64498.898947,"MULTILINESTRING ((758390.816 4094829.522, 7583..."
3019,255391103,Yosemite,BONITA,12kV,3742,3842,63686.393390,"MULTILINESTRING ((749035.452 4094939.762, 7490..."
3020,255451102,Kern,CAL WATER,12kV,10520,370,97943.786622,"MULTILINESTRING ((876103.782 3926133.744, 8762..."
3021,255451103,Kern,CAL WATER,12kV,10380,300,72464.525600,"MULTILINESTRING ((874170.252 3926169.744, 8741..."


In [18]:
# Function to import ICA data of specific feeder_id
def get_feeder_ICA_df_from_parquet(feeder_id):
    """
    get_feeder_ICA_df_from_parquet

    :feeder_id: is the id of feeder for which you want the ICA data
    :return:
    """
    filename = feeder_id + ".parquet"
    file_path = DATA_DIR / "cleanedGridData" / "ICA_Load_CLEAN_PARQUET_v4" / filename
    
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    df = pd.read_parquet(file_path)
    print(f"Loaded {filename} with {len(df)} rows and {len(df.columns)} columns.")
    return df

feeder_ICA_df = get_feeder_ICA_df_from_parquet("012011109")
display(feeder_ICA_df.head())

Loaded 012011109.parquet with 214272 rows and 9 columns.


,line_section_id,month,hour,IC10_Thermal_KW,IC10_Voltage_KW,IC90_Thermal_KW,IC90_Voltage_KW,feeder_id,division
0,3034382,1,0,3780.0,4300.0,3360.0,4300.0,012011109,East Bay
1,3034382,1,1,3840.0,4300.0,3470.0,4300.0,012011109,East Bay
2,3034382,1,2,3900.0,4300.0,3480.0,4290.0,012011109,East Bay
3,3034382,1,3,3900.0,4300.0,3510.0,4300.0,012011109,East Bay
4,3034382,1,4,3900.0,4300.0,3500.0,4300.0,012011109,East Bay


# Plotting Functions

# Network Analsysis

In [ ]:
# Create list of all feeders in east bay division
